# 🚀 Training Sugoi V5 Custom (Light Novel NMT Fine-Tuning)
Notebook ini melatih ulang model MarianMT/Sugoi dengan dataset light novel Gold Standard (`train.json` & `val.json`), lalu otomatis mengonversinya menjadi `model.bin` CTranslate2 yang super kencang untuk dipakai di Novel TL.

> **Catatan**: Pastikan Runtime diatur ke **GPU T4**: Menu `Runtime` -> `Change runtime type` -> pilih `T4 GPU`.

In [ ]:
# 1. Periksa GPU
!nvidia-smi

In [ ]:
# 2. Install library yang dibutuhkan
!pip install -q transformers datasets sentencepiece accelerate evaluate ctranslate2

In [ ]:
# 3. Upload dataset (train.json dan val.json)
from google.colab import files
import os

os.makedirs("dataset", exist_ok=True)
print("Silakan upload train.json dan val.json dari folder data/dataset/ Anda:")
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, os.path.join("dataset", filename))

!ls -lh dataset/

In [ ]:
# 4. Training Model (Helsinki-NLP/opus-mt-ja-en)
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset

model_checkpoint = "Helsinki-NLP/opus-mt-ja-en"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

dataset = load_dataset("json", data_files={
    "train": "dataset/train.json",
    "validation": "dataset/val.json"
})

def preprocess_function(examples):
    inputs = examples["ja"]
    targets = examples["en"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./output_sugoi_custom",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    tokenizer=tokenizer
)

print("Memulai Fine-Tuning...")
trainer.train()

# Simpan model PyTorch
final_model_dir = "./output_sugoi_custom/final_model"
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("Training Selesai! Model tersimpan di:", final_model_dir)

In [ ]:
# 5. Konversi Model ke CTranslate2 (INT8)
!ct2-transformers-converter --model ./output_sugoi_custom/final_model --output_dir ./ct2_sugoi_custom --quantization int8 --force
!ls -lh ./ct2_sugoi_custom

In [ ]:
# 6. Download file model.bin hasil latihan ke komputer Anda
from google.colab import files
print("Mengunduh model.bin baru...")
files.download("./ct2_sugoi_custom/model.bin")